# ZIP File Flow

Notebook nay dung `diagnostic_controller.py` + Tampermonkey script hien tai de chay flow 4 agent trong moi truong ZIP kin.

Flow mac dinh: `C -> A -> T -> B -> C -> ...`.

Truoc khi `Run All`:
1. Start server: `python .\\diagnostic_controller.py`
2. Mo 4 tab ChatGPT va set role: `C`, `A`, `T`, `B`
3. Paste `tampermonkey.js` moi nhat va refresh cac tab
4. Upload `repo.zip` va sample file vao tab `A`
5. Sua `INITIAL_TASK` neu can, sau do `Run All`


In [3]:
# BLOCK 1: imports + admin helpers
import json
import os
import re
import time
import urllib.parse
import urllib.request
import urllib.error
from pathlib import Path
from IPython.display import clear_output

BASE_URL = "http://127.0.0.1:8500"
ACTIVE_ROLES = ["C", "A", "T", "B"]
PROMPTS_DIR = Path("prompts")
RUN_LOG_PATH = Path("zipfile_flow_run_log.json")

# Proxy settings (co the set tu terminal: $env:HTTP_PROXY / $env:HTTPS_PROXY)
HTTP_PROXY = os.environ.get("HTTP_PROXY", "")
HTTPS_PROXY = os.environ.get("HTTPS_PROXY", "")
USE_PROXY = bool(HTTP_PROXY or HTTPS_PROXY)

# Opener khong proxy cho localhost
_NO_PROXY_OPENER = urllib.request.build_opener(urllib.request.ProxyHandler({}))

# Opener co proxy cho endpoint ben ngoai (neu can)
if USE_PROXY:
    _PROXY_OPENER = urllib.request.build_opener(
        urllib.request.ProxyHandler({
            "http": HTTP_PROXY or HTTPS_PROXY,
            "https": HTTPS_PROXY or HTTP_PROXY,
        })
    )
else:
    _PROXY_OPENER = _NO_PROXY_OPENER

def _decode_error_body(err):
    try:
        body = err.read()
        if not body:
            return ""
        return body.decode("utf-8", errors="replace")
    except Exception:
        return ""

def _select_opener(url):
    parsed = urllib.parse.urlparse(url)
    host = (parsed.hostname or "").lower()
    # Local admin API phai di thang, khong qua proxy
    if host in {"127.0.0.1", "localhost", "::1"}:
        return _NO_PROXY_OPENER
    return _PROXY_OPENER

def http_json(method, path, payload=None, timeout=300, retries=2, retry_wait_s=1.0):
    url = f"{BASE_URL}{path}"
    data = None
    headers = {}
    if payload is not None:
        data = json.dumps(payload, ensure_ascii=False).encode("utf-8")
        headers["Content-Type"] = "application/json"

    req = urllib.request.Request(url, data=data, headers=headers, method=method)

    last_exc = None
    opener = _select_opener(url)
    for attempt in range(retries + 1):
        try:
            with opener.open(req, timeout=timeout) as resp:
                text = resp.read().decode("utf-8")
                return json.loads(text) if text else {}
        except urllib.error.HTTPError as e:
            body = _decode_error_body(e)
            retryable = e.code in (502, 503, 504)
            last_exc = RuntimeError(
                f"HTTP {e.code} {e.reason} when calling {method} {path}. "
                f"Response body: {body[:500]}"
            )
            if retryable and attempt < retries:
                time.sleep(retry_wait_s * (attempt + 1))
                continue
            raise last_exc from e
        except urllib.error.URLError as e:
            last_exc = RuntimeError(f"Cannot connect to {url}: {e.reason}")
            if attempt < retries:
                time.sleep(retry_wait_s * (attempt + 1))
                continue
            raise last_exc from e

    raise last_exc or RuntimeError(f"Unknown error when calling {method} {path}")

def get_config():
    return http_json("GET", "/api/admin/config")["config"]

def update_config(updates):
    return http_json("POST", "/api/admin/config", {"config": updates})["config"]

def role_snapshot(role):
    return http_json("GET", f"/api/admin/role/{urllib.parse.quote(role)}")

def event_tail(role="", contains="", command_id="", limit=50):
    params = urllib.parse.urlencode({
        "role": role,
        "contains": contains,
        "command_id": command_id,
        "limit": limit,
    })
    return http_json("GET", f"/api/admin/events?{params}")

def send_command(role, action, payload=None):
    return http_json("POST", "/api/admin/command", {
        "role": role,
        "action": action,
        "payload": payload or {},
    })["command"]

def wait_command(command_id, timeout=300, print_every=1.0):
    started = time.time()
    last_print = 0
    while True:
        data = http_json("GET", f"/api/admin/command/{command_id}")
        now = time.time()
        if now - last_print >= print_every:
            print(f"[{time.strftime('%H:%M:%S')}] cmd={command_id[:8]} status={data['status']} done={data['done']}")
            last_print = now
        if data["done"]:
            return data["result"]
        if timeout and now - started > timeout:
            raise TimeoutError(f"Timeout waiting for command_id={command_id}")
        time.sleep(0.2)

def run_command(role, action, payload=None, timeout=300, print_every=1.0):
    cmd = send_command(role, action, payload)
    print(f"{action} -> role={role}, command_id={cmd['command_id']}")
    return wait_command(cmd["command_id"], timeout=timeout, print_every=print_every)

def dom_summary(dom):
    messages = dom.get("messages") or {}
    counts = messages.get("counts") if isinstance(messages, dict) else None
    return {
        "composer": bool(dom.get("composer")),
        "composer_text_len": dom.get("composer_text_len"),
        "send_enabled": dom.get("send_enabled"),
        "stop_visible": dom.get("stop_visible"),
        "voice_visible": dom.get("voice_visible"),
        "selection_strategy": dom.get("selection_strategy"),
        "matched_selector": dom.get("matched_selector"),
        "message_counts": counts,
    }

print("Proxy mode:", "ON" if USE_PROXY else "OFF")
if USE_PROXY:
    print("HTTP_PROXY:", HTTP_PROXY)
    print("HTTPS_PROXY:", HTTPS_PROXY)

# Khong de notebook vo ngay neu server dang khoi dong/chua san sang
try:
    print(json.dumps(get_config(), ensure_ascii=False, indent=2))
except Exception as e:
    print("Khong lay duoc config tu diagnostic_controller.")
    print("Loi:", e)
    print("Kiem tra: da chay `python .\\diagnostic_controller.py` va server da ready chua.")


Proxy mode: ON
HTTP_PROXY: http://in-proxy-o.denso.co.jp:8080
HTTPS_PROXY: http://in-proxy-o.denso.co.jp:8080
{
  "poll_ms": 800,
  "sync_debounce_ms": 1200,
  "wait_loop_interval_ms": 500,
  "action_delay_min_ms": 3000,
  "action_delay_max_ms": 5000,
  "send_delay_min_ms": 2000,
  "send_delay_max_ms": 5000,
  "role_switch_delay_min_s": 3,
  "role_switch_delay_max_s": 5,
  "composer_stable_samples": 6,
  "composer_stable_sample_ms": 300,
  "assistant_quiet_ms": 2500,
  "send_accept_timeout_ms": 10000,
  "send_accept_poll_ms": 400,
  "assistant_force_sync_quiet_ms": 5000,
  "assistant_post_stop_timeout_ms": 15000,
  "report_wait_every_ms": 1500,
  "max_button_dump": 80,
  "auto_reload_on_assistant_timeout": true,
  "reload_after_timeout_ms": 1500
}


In [4]:
# BLOCK 1 TEST: comprehensive smoke test for admin helpers
test_report = {
    "base_url": BASE_URL,
    "checks": {},
    "errors": [],
}

# 1) get_config
try:
    cfg = get_config()
    test_report["checks"]["get_config"] = {
        "ok": isinstance(cfg, dict),
        "keys": sorted(list(cfg.keys()))[:10],
    }
except Exception as e:
    test_report["checks"]["get_config"] = {"ok": False}
    test_report["errors"].append(f"get_config: {e}")

# 2) update_config round-trip (no-op style)
try:
    old_poll = cfg.get("poll_ms", 800) if isinstance(cfg, dict) else 800
    new_cfg = update_config({"poll_ms": old_poll})
    test_report["checks"]["update_config"] = {
        "ok": isinstance(new_cfg, dict) and new_cfg.get("poll_ms") == old_poll,
        "poll_ms": new_cfg.get("poll_ms"),
    }
except Exception as e:
    test_report["checks"]["update_config"] = {"ok": False}
    test_report["errors"].append(f"update_config: {e}")

# 3) role snapshots
role_checks = {}
for role in ACTIVE_ROLES:
    try:
        snap = role_snapshot(role)
        role_checks[role] = {
            "ok": True,
            "status": snap.get("status"),
            "sessions": snap.get("sessions"),
        }
    except Exception as e:
        role_checks[role] = {"ok": False, "error": str(e)}
test_report["checks"]["role_snapshot"] = role_checks

# 4) event tail
try:
    tail = event_tail(limit=5)
    events = tail.get("events", []) if isinstance(tail, dict) else []
    test_report["checks"]["event_tail"] = {
        "ok": isinstance(events, list),
        "event_count": len(events),
    }
except Exception as e:
    test_report["checks"]["event_tail"] = {"ok": False}
    test_report["errors"].append(f"event_tail: {e}")

# 5) command API basic (create + status read)
try:
    cmd = send_command("C", "PROBE", {})
    cmd_id = cmd.get("command_id")
    status_obj = http_json("GET", f"/api/admin/command/{cmd_id}") if cmd_id else {}
    test_report["checks"]["command_api"] = {
        "ok": bool(cmd_id) and isinstance(status_obj, dict),
        "command_id": cmd_id,
        "status": status_obj.get("status"),
        "done": status_obj.get("done"),
    }
except Exception as e:
    test_report["checks"]["command_api"] = {"ok": False}
    test_report["errors"].append(f"command_api: {e}")

print(json.dumps(test_report, ensure_ascii=False, indent=2))

{
  "base_url": "http://127.0.0.1:8500",
  "checks": {
    "get_config": {
      "ok": true,
      "keys": [
        "action_delay_max_ms",
        "action_delay_min_ms",
        "assistant_force_sync_quiet_ms",
        "assistant_post_stop_timeout_ms",
        "assistant_quiet_ms",
        "auto_reload_on_assistant_timeout",
        "composer_stable_sample_ms",
        "composer_stable_samples",
        "max_button_dump",
        "poll_ms"
      ]
    },
    "update_config": {
      "ok": true,
      "poll_ms": 800
    },
    "role_snapshot": {
      "C": {
        "ok": true,
        "status": "OFFLINE",
        "sessions": []
      },
      "A": {
        "ok": true,
        "status": "ONLINE",
        "sessions": [
          "/"
        ]
      },
      "T": {
        "ok": true,
        "status": "OFFLINE",
        "sessions": []
      },
      "B": {
        "ok": true,
        "status": "OFFLINE",
        "sessions": []
      }
    },
    "event_tail": {
      "ok": true,
      

In [ ]:
# BLOCK 2: prompt files + run config
INITIAL_TASK = """Please coordinate the multi-agent workflow for this task.

Shared workspace:
- MCP: thinkbook-tail-mcp
- ROOT_PATH: C:\\Users\\admin\\Desktop\\gpt_mauto\\gpt\\source_task\\aspice-automotive-development

Task goal:
Improve/fix the Excel parsing workflow in the repository so it can:
1. Extract the complete textual content from Excel files.
2. Extract all embedded images from Excel files.
3. Preserve image correctness and quality.
4. Produce auditable outputs for both text and image extraction.
5. Provide enough evidence for testing, review, and truth-checking.

Important requirements:
- Focus on Excel parsing.
- Text extraction must cover the full workbook as much as technically possible.
- Image extraction must be complete and auditable.
- Extracted images must be validated for existence, readability, format, dimensions, byte size, corruption, missing/duplicate cases, and abnormal quality loss.
- The Tester must independently analyze output images, not merely trust logs.
- The Reviewer must check robustness against common Excel variants and silent data loss.
- The Coordinator must truth-check all agent claims before finishing.

Expected flow:
C -> A -> T -> B -> C, looping back to A if T or B fails.

Start by routing the task to A with concrete instructions.
"""

COORDINATOR_TIMEOUT_S = 300
WORKER_TIMEOUT_S = 1800

PROMPTS_CACHE = {}
PERSONA_LOADED = {role: False for role in ACTIVE_ROLES}

def load_prompt(role):
    txt_path = PROMPTS_DIR / f"{role}.txt"
    json_path = PROMPTS_DIR / f"{role}.json"
    if txt_path.exists():
        return txt_path.read_text(encoding="utf-8").strip()
    if json_path.exists():
        data = json.loads(json_path.read_text(encoding="utf-8"))
        return str(data.get("prompt", "")).strip()
    raise FileNotFoundError(f"Missing prompt file for role {role}: {txt_path} or {json_path}")

def reload_prompts():
    global PROMPTS_CACHE
    PROMPTS_CACHE = {role: load_prompt(role) for role in ACTIVE_ROLES}
    return PROMPTS_CACHE

def reset_persona_loaded():
    global PERSONA_LOADED
    PERSONA_LOADED = {role: False for role in ACTIVE_ROLES}

prompts = reload_prompts()
print({role: len(text) for role, text in prompts.items()})


In [ ]:
# BLOCK 3: live config tuning
cfg = update_config({
    "action_delay_min_ms": 3000,
    "action_delay_max_ms": 5000,
    "send_delay_min_ms": 2000,
    "send_delay_max_ms": 5000,
    "assistant_quiet_ms": 2500,
    "assistant_force_sync_quiet_ms": 5000,
    "assistant_post_stop_timeout_ms": 15000,
    "auto_reload_on_assistant_timeout": True,
    "reload_after_timeout_ms": 1500,
})
print(json.dumps(cfg, ensure_ascii=False, indent=2))


In [ ]:
# BLOCK 4: flow helpers
def wait_role_ready(role, attempts=30, sleep_s=1):
    for i in range(attempts):
        snap = role_snapshot(role)
        print(f"[{role}] check {i}: status={snap['status']} sessions={snap['sessions']}")
        if snap["status"] != "OFFLINE" or snap.get("dom_info"):
            print(json.dumps(dom_summary(snap.get("dom_info", {})), ensure_ascii=False, indent=2))
            return snap
        time.sleep(sleep_s)
    raise RuntimeError(f"Role {role} is still offline")

def compose_role_message(role, task_text, from_coordinator=False):
    if PERSONA_LOADED.get(role, False):
        return task_text
    prefix = PROMPTS_CACHE[role]
    if from_coordinator:
        msg = f"{prefix}\n\n{'=' * 40}\n[LENH TU COORDINATOR]\n{task_text}"
    else:
        msg = f"{prefix}\n\n{'=' * 40}\n{task_text}"
    PERSONA_LOADED[role] = True
    return msg

def parse_coordinator_decision(text):
    if "TASK COMPLETE" in (text or ""):
        return {
            "target": "FINISH",
            "reason": "Coordinator emitted TASK COMPLETE",
            "message": "TASK COMPLETE",
        }

    json_str = ""
    fenced = re.search(r"```(?:json)?\s*(.*?)\s*```", text or "", re.DOTALL | re.IGNORECASE)
    if fenced:
        json_str = fenced.group(1)
    else:
        bracket = re.search(r"(\{.*\})", text or "", re.DOTALL)
        if bracket:
            json_str = bracket.group(1)

    if json_str:
        try:
            return json.loads(json_str)
        except json.JSONDecodeError:
            pass

    return {
        "target": "HUMAN",
        "reason": "Coordinator did not return valid JSON",
        "message": text or "",
    }

def run_role_turn(role, prompt_text, timeout=WORKER_TIMEOUT_S, print_every=2.0):
    result = {}
    result["probe"] = run_command(role, "PROBE", timeout=60, print_every=print_every)
    result["set_prompt"] = run_command(role, "SET_PROMPT", {
        "text": prompt_text,
        "method": "auto",
        "samples": 6,
        "sample_ms": 300,
    }, timeout=120, print_every=print_every)
    result["find_send"] = run_command(role, "FIND_SEND", timeout=60, print_every=print_every)
    result["click_send"] = run_command(role, "CLICK_SEND", timeout=60, print_every=print_every)
    if result["click_send"]["state"] != "SEND_ACCEPTED":
        raise RuntimeError(
            f"{role} CLICK_SEND did not reach SEND_ACCEPTED: state={result['click_send']['state']} "
            f"result={json.dumps(result['click_send'].get('result', {}), ensure_ascii=False)}"
        )
    result["assistant"] = run_command(role, "WAIT_ASSISTANT_DONE", timeout=timeout, print_every=print_every)
    if result["assistant"]["state"] != "ASSISTANT_DONE":
        raise RuntimeError(
            f"{role} WAIT_ASSISTANT_DONE did not finish cleanly: state={result['assistant']['state']} "
            f"result={json.dumps(result['assistant'].get('result', {}), ensure_ascii=False)}"
        )
    result["sync"] = run_command(role, "SYNC_TRANSCRIPT", {"reason": "zipfile_flow_post_turn_sync"}, timeout=60, print_every=print_every)
    result["response_text"] = (result["assistant"].get("text") or "").strip()
    return result

def summarize_turn(role, item):
    print(json.dumps({
        "role": role,
        "probe": item["probe"]["state"],
        "set_prompt": item["set_prompt"]["state"],
        "find_send": item["find_send"]["state"],
        "selection_strategy": (item["find_send"].get("result") or {}).get("selection_strategy"),
        "matched_selector": (item["find_send"].get("result") or {}).get("matched_selector"),
        "click_send": item["click_send"]["state"],
        "send_reasons": (item["click_send"].get("result") or {}).get("reasons"),
        "assistant": item["assistant"]["state"],
        "response_preview": item["response_text"][:300],
    }, ensure_ascii=False, indent=2))


In [ ]:
# BLOCK 5: run full coordinator flow
reload_prompts()
reset_persona_loaded()

for role in ACTIVE_ROLES:
    wait_role_ready(role, attempts=20, sleep_s=1)

current_output = f"Sep (Human) giao viec: {INITIAL_TASK}"
last_actor = "HUMAN"
turn = 1
run_log = []

while True:
    clear_output(wait=True)
    print("=" * 90)
    print(f"TURN {turn}: COORDINATOR C")
    print("=" * 90)
    print("Last actor:", last_actor)
    print(current_output[:1500])
    print("=" * 90)

    prompt_for_c = f"[BAO CAO TU {last_actor}]\n{current_output}\n\nHay quyet dinh buoc tiep theo."
    c_message = compose_role_message("C", prompt_for_c, from_coordinator=False)
    c_turn = run_role_turn("C", c_message, timeout=COORDINATOR_TIMEOUT_S, print_every=2.0)
    c_text = c_turn["response_text"]
    decision = parse_coordinator_decision(c_text)

    target = decision.get("target", "HUMAN")
    reason = decision.get("reason", "")
    message = decision.get("message", "")

    clear_output(wait=True)
    print("=" * 90)
    print(f"COORDINATOR DECISION - TURN {turn}")
    print("=" * 90)
    print("Target :", target)
    print("Reason :", reason)
    print("Message:", message[:800] + ("..." if len(message) > 800 else ""))
    print("=" * 90)

    log_item = {
        "turn": turn,
        "last_actor": last_actor,
        "current_output": current_output,
        "coordinator_response": c_text,
        "decision": decision,
    }

    if target == "FINISH" or message == "TASK COMPLETE":
        log_item["status"] = "TASK COMPLETE"
        run_log.append(log_item)
        RUN_LOG_PATH.write_text(json.dumps(run_log, ensure_ascii=False, indent=2), encoding="utf-8")
        print("\nTASK COMPLETE")
        print(f"Saved log: {RUN_LOG_PATH.resolve()}")
        break

    if target == "HUMAN":
        log_item["status"] = "HUMAN_NEEDED"
        run_log.append(log_item)
        RUN_LOG_PATH.write_text(json.dumps(run_log, ensure_ascii=False, indent=2), encoding="utf-8")
        raise RuntimeError("Coordinator requested HUMAN input. Inspect the last decision in run_log.")

    if target not in ACTIVE_ROLES:
        log_item["status"] = "INVALID_TARGET"
        run_log.append(log_item)
        RUN_LOG_PATH.write_text(json.dumps(run_log, ensure_ascii=False, indent=2), encoding="utf-8")
        raise RuntimeError(f"Invalid target from coordinator: {target}")

    worker_message = compose_role_message(target, message, from_coordinator=True)
    worker_turn = run_role_turn(target, worker_message, timeout=WORKER_TIMEOUT_S, print_every=2.0)
    summarize_turn(target, worker_turn)

    current_output = worker_turn["response_text"] or f"[{target}] khong tra ve noi dung."
    last_actor = target
    log_item["worker_role"] = target
    log_item["worker_response"] = current_output
    log_item["worker_turn"] = worker_turn
    log_item["status"] = "CONTINUE"
    run_log.append(log_item)
    RUN_LOG_PATH.write_text(json.dumps(run_log, ensure_ascii=False, indent=2), encoding="utf-8")
    turn += 1

print("Done")


---

In [1]:

# ============================================================
# STANDALONE BLOCK A: imports + helpers (chạy độc lập, không cần Cell 2)
# ============================================================
import json
import os
import time
import urllib.parse
import urllib.request
import urllib.error
from pathlib import Path

BASE_URL = "http://127.0.0.1:8500"
ACTIVE_ROLES = ["C", "A", "T", "B"]
PROMPTS_DIR = Path("prompts")

PROBE_TIMEOUT_S     = 60
SET_PROMPT_TIMEOUT_S = 120
CLICK_TIMEOUT_S     = 60
ASSISTANT_TIMEOUT_S = 1800
SYNC_TIMEOUT_S      = 60

# Proxy: dung cho external, bypass cho localhost
_HTTP_PROXY  = os.environ.get("HTTP_PROXY", "")
_HTTPS_PROXY = os.environ.get("HTTPS_PROXY", "")
_NO_PROXY_OPENER = urllib.request.build_opener(urllib.request.ProxyHandler({}))
if _HTTP_PROXY or _HTTPS_PROXY:
    _EXT_OPENER = urllib.request.build_opener(
        urllib.request.ProxyHandler({
            "http":  _HTTP_PROXY  or _HTTPS_PROXY,
            "https": _HTTPS_PROXY or _HTTP_PROXY,
        })
    )
else:
    _EXT_OPENER = _NO_PROXY_OPENER

def _select_opener(url):
    host = (urllib.parse.urlparse(url).hostname or "").lower()
    if host in {"127.0.0.1", "localhost", "::1"}:
        return _NO_PROXY_OPENER
    return _EXT_OPENER

def http_json(method, path, payload=None, timeout=300, retries=2, retry_wait_s=1.0):
    url = f"{BASE_URL}{path}"
    data, headers = None, {}
    if payload is not None:
        data = json.dumps(payload, ensure_ascii=False).encode("utf-8")
        headers["Content-Type"] = "application/json"
    req    = urllib.request.Request(url, data=data, headers=headers, method=method)
    opener = _select_opener(url)
    last_exc = None
    for attempt in range(retries + 1):
        try:
            with opener.open(req, timeout=timeout) as resp:
                text = resp.read().decode("utf-8")
                return json.loads(text) if text else {}
        except urllib.error.HTTPError as e:
            try:
                body = e.read().decode("utf-8", errors="replace")[:400]
            except Exception:
                body = ""
            retryable = e.code in (502, 503, 504)
            last_exc = RuntimeError(f"HTTP {e.code} {e.reason} {method} {path} body={body}")
            if retryable and attempt < retries:
                time.sleep(retry_wait_s * (attempt + 1))
                continue
            raise last_exc from e
        except urllib.error.URLError as e:
            last_exc = RuntimeError(f"Cannot connect {url}: {e.reason}")
            if attempt < retries:
                time.sleep(retry_wait_s * (attempt + 1))
                continue
            raise last_exc from e
    raise last_exc or RuntimeError(f"Unknown error {method} {path}")

def send_command(role, action, payload=None):
    return http_json("POST", "/api/admin/command", {
        "role": role, "action": action, "payload": payload or {},
    })["command"]

def wait_command(command_id, timeout=300, print_every=2.0):
    started = time.time()
    last_print = 0.0
    while True:
        data = http_json("GET", f"/api/admin/command/{command_id}")
        now  = time.time()
        if now - last_print >= print_every:
            print(f"[{time.strftime('%H:%M:%S')}] cmd={command_id[:8]} status={data['status']} done={data['done']}")
            last_print = now
        if data["done"]:
            return data["result"]
        if timeout and (now - started) > timeout:
            raise TimeoutError(f"Timeout waiting command_id={command_id}")
        time.sleep(0.2)

def run_command(role, action, payload=None, timeout=300, print_every=2.0):
    cmd = send_command(role, action, payload)
    print(f"{action} -> role={role}, command_id={cmd['command_id']}")
    return wait_command(cmd["command_id"], timeout=timeout, print_every=print_every)

def try_reset_page(role):
    for action in ["RESET_PAGE", "RELOAD_PAGE", "HARD_RELOAD", "RELOAD"]:
        try:
            res = run_command(role, action, timeout=90, print_every=1.0)
            print(f"Reset OK: action={action}, state={res.get('state')}")
            return {"ok": True, "action": action, "result": res}
        except Exception as e:
            print(f"Reset skip: {action} -> {e}")
    return {"ok": False, "action": None, "result": None}

def load_prompt(role):
    txt_path  = PROMPTS_DIR / f"{role}.txt"
    json_path = PROMPTS_DIR / f"{role}.json"
    if txt_path.exists():
        return txt_path.read_text(encoding="utf-8").strip()
    if json_path.exists():
        data = json.loads(json_path.read_text(encoding="utf-8"))
        return str(data.get("prompt", "")).strip()
    raise FileNotFoundError(f"Missing prompt file for role {role}")

# Quick connectivity check
try:
    cfg = http_json("GET", "/api/admin/config")["config"]
    print("Server OK, poll_ms =", cfg.get("poll_ms"))
except Exception as e:
    print("Server NOT reachable:", e)
    print("Chay: .venv\\Scripts\\python.exe .\\diagnostic_controller.py")


Server OK, poll_ms = 800


In [11]:

# ============================================================
# STANDALONE BLOCK B: run_agent_prompt + ask()
# Phu thuoc: phai chay Standalone Block A truoc
# ============================================================

def open_new_chat(role, wait_s=3.0):
    """Mo conversation moi tren tab role. Thu NEW_CHAT -> NAVIGATE_NEW -> fallback RELOAD."""
    for action in ["NEW_CHAT", "NAVIGATE_NEW"]:
        try:
            res = run_command(role, action, timeout=30, print_every=1.0)
            print(f"[new_chat] {action} OK: state={res.get('state')}")
            if wait_s > 0:
                time.sleep(wait_s)
            return {"ok": True, "action": action, "result": res}
        except Exception as e:
            print(f"[new_chat] {action} not supported: {e}")
    for action in ["RELOAD_PAGE", "HARD_RELOAD", "RELOAD"]:
        try:
            res = run_command(role, action, timeout=30, print_every=1.0)
            print(f"[new_chat] fallback {action} OK: state={res.get('state')}")
            if wait_s > 0:
                time.sleep(wait_s)
            return {"ok": True, "action": action, "result": res}
        except Exception as e:
            print(f"[new_chat] fallback {action} skip: {e}")
    print(f"[new_chat] WARNING: khong mo duoc conversation moi cho tab {role}.")
    return {"ok": False, "action": None, "result": None}


def run_agent_prompt(
    role,
    prompt_text,
    do_reset=False,
    new_chat=False,
    new_chat_wait_s=3.0,
    timeout_s=ASSISTANT_TIMEOUT_S,
    print_every=2.0,
):
    """
    Full flow cho 1 turn: PROBE -> SET_PROMPT -> FIND_SEND -> CLICK_SEND
    -> WAIT_ASSISTANT_DONE -> SYNC_TRANSCRIPT.

    do_reset  : F5 reload tab truoc khi gui (giu nguyen conversation)
    new_chat  : Mo conversation MOI truoc khi gui
    """
    if role not in ACTIVE_ROLES:
        raise ValueError(f"Invalid role={role!r}, phai la mot trong {ACTIVE_ROLES}")

    if new_chat:
        open_new_chat(role, wait_s=new_chat_wait_s)
    elif do_reset:
        try_reset_page(role)

    result = {}
    result["probe"] = run_command(role, "PROBE", timeout=PROBE_TIMEOUT_S, print_every=print_every)
    result["set_prompt"] = run_command(
        role, "SET_PROMPT",
        {"text": prompt_text, "method": "auto", "samples": 6, "sample_ms": 300},
        timeout=SET_PROMPT_TIMEOUT_S, print_every=print_every,
    )
    result["find_send"]  = run_command(role, "FIND_SEND",  timeout=PROBE_TIMEOUT_S,  print_every=print_every)
    result["click_send"] = run_command(role, "CLICK_SEND", timeout=CLICK_TIMEOUT_S,  print_every=print_every)

    if result["click_send"].get("state") != "SEND_ACCEPTED":
        raise RuntimeError(
            f"CLICK_SEND failed: state={result['click_send'].get('state')}, "
            f"result={json.dumps(result['click_send'].get('result', {}), ensure_ascii=False)}"
        )

    result["assistant"] = run_command(
        role, "WAIT_ASSISTANT_DONE",
        timeout=timeout_s, print_every=print_every,
    )

    if result["assistant"].get("state") != "ASSISTANT_DONE":
        raise RuntimeError(
            f"WAIT_ASSISTANT_DONE failed: state={result['assistant'].get('state')}, "
            f"result={json.dumps(result['assistant'].get('result', {}), ensure_ascii=False)}"
        )

    result["sync"] = run_command(
        role, "SYNC_TRANSCRIPT", {"reason": "ask_single_turn"},
        timeout=SYNC_TIMEOUT_S, print_every=print_every,
    )
    result["response_text"] = (result["assistant"].get("text") or "").strip()
    return result


def ask(
    role,
    prompt,
    persona_file=True,
    from_coordinator=True,
    do_reset=False,
    new_chat=False,
    timeout_s=ASSISTANT_TIMEOUT_S,
):
    """
    Gui prompt cho agent, nhan ve response text.

    role             : "A", "T", "B", "C"
    prompt           : noi dung gui cho agent
    persona_file     : them noi dung prompts/<role>.txt vao truoc prompt
    from_coordinator : them header [LENH TU COORDINATOR]
    do_reset         : F5 reload tab (giu conversation cu)
    new_chat         : Mo conversation MOI (uu tien hon do_reset)
    timeout_s        : timeout doi assistant (giay)

    Vi du:
      response = ask("A", "Phan tich file Excel.")              # gui thang
      response = ask("A", "Phan tich file Excel.", do_reset=True)  # F5 truoc
      response = ask("A", "Phan tich file Excel.", new_chat=True)  # new chat truoc
    """
    if persona_file:
        persona = load_prompt(role)
        header  = "[LENH TU COORDINATOR]\n" if from_coordinator else ""
        full_prompt = f"{persona}\n\n{'=' * 40}\n{header}{prompt}"
    else:
        full_prompt = prompt

    return run_agent_prompt(
        role=role,
        prompt_text=full_prompt,
        do_reset=do_reset,
        new_chat=new_chat,
        timeout_s=timeout_s,
    )["response_text"]


In [13]:

# new_chat=True => mo conversation moi truoc khi gui
a = ask("A", """phân tích toàn bộ đoạn code sau:
        
        
        """, persona_file=False, from_coordinator=False, new_chat=False)


PROBE -> role=A, command_id=96d017fc-e33e-41e9-8d6f-9ab95f947710
[11:02:33] cmd=96d017fc status=PENDING done=False
SET_PROMPT -> role=A, command_id=0ef5eacc-c3e2-4ab3-988f-252c13d3643b
[11:02:34] cmd=0ef5eacc status=PENDING done=False
[11:02:36] cmd=0ef5eacc status=DELIVERED done=False
[11:02:38] cmd=0ef5eacc status=DELIVERED done=False
FIND_SEND -> role=A, command_id=0e1f0bfd-62bd-490a-8493-25ac48b83691
[11:02:40] cmd=0e1f0bfd status=PENDING done=False
CLICK_SEND -> role=A, command_id=6248b045-1fc6-4409-9a25-5c663da7a9a7
[11:02:41] cmd=6248b045 status=PENDING done=False
[11:02:44] cmd=6248b045 status=DELIVERED done=False
[11:02:46] cmd=6248b045 status=DELIVERED done=False
WAIT_ASSISTANT_DONE -> role=A, command_id=eeaddb0a-69c5-42fd-875b-987e064e4649
[11:02:47] cmd=eeaddb0a status=PENDING done=False
[11:02:49] cmd=eeaddb0a status=DELIVERED done=False
[11:02:51] cmd=eeaddb0a status=ASSISTANT_TEXT_CHANGED done=False
[11:02:54] cmd=eeaddb0a status=ASSISTANT_TEXT_CHANGED done=False
[11:02:

In [15]:
len(a)

7570

In [14]:
a

'Đây là một notebook Jupyter dùng để điều phối tự động nhiều tab ChatGPT như một hệ thống multi-agent, thông qua:\n\ndiagnostic_controller.py (server local tại 127.0.0.1:8500)\n\nTampermonkey script chạy trong browser\n\n4 role:\n\nC = Coordinator\n\nA = Analyst/Developer\n\nT = Tester\n\nB = Reviewer\n\nFlow:\n\nHuman\n  ↓\nC (Coordinator)\n  ↓\nA (Developer)\n  ↓\nT (Tester)\n  ↓\nB (Reviewer)\n  ↓\nC\n  ↓\n(A/T/B tiếp tục nếu cần)\n1. Mục tiêu tổng thể của notebook\n\nNotebook này không làm việc trực tiếp với source code.\n\nNó làm nhiệm vụ:\n\nNotebook\n    ↓ REST API\ndiagnostic_controller.py\n    ↓\nTampermonkey\n    ↓\n4 tab ChatGPT\n\nTức là:\n\nNotebook gửi command:\n\nPython\nSET_PROMPT\nCLICK_SEND\nWAIT_ASSISTANT_DONE\nSYNC_TRANSCRIPT\n\ncho từng tab ChatGPT.\n\nNó biến ChatGPT thành:\n\nAgent A\nAgent T\nAgent B\nAgent C\n\nvà tự động trao đổi với nhau.\n\n2. Kiến trúc\nzipfile_flow.ipynb\n        |\n        v\ndiagnostic_controller.py\n        |\n        v\nREST API :8500\